# Лабораторная работа №1
## Численные методы алгебры

**Цель:** реализовать прямые и итерационные методы решения СЛАУ, методы решения нелинейных уравнений и систем; исследовать точность, обусловленность и сходимость.


### Правила выполнения
- Сделайте собственную копию блокнота. Заполняйте ячейки с `TODO`, добавляйте эксперименты и выводы непосредственно в него.
- XML-модели, визуализация и регулятор предоставлены готовыми. Их переписывать не требуется.
- В собственных решателях запрещены `np.linalg.solve`, `inv`, `lu`, `cholesky` и аналогичные готовые решатели/разложения. Они разрешены только для эталонных проверок. `norm`, матричные операции и вычисление числа обусловленности разрешены.
- Использование агентов разрешено. На защите необходимо объяснить код, предсказать результат изменения параметров и провести эксперимент.
- Задания обозначены **А1–А5, Б1–Б3**.

## Подготовка
При первой установке выполните следующую ячейку. Затем перезапустите сеанс через меню «Среда выполнения → Перезапустить сеанс» и продолжите с импортов. Если MuJoCo уже импортировался с другим OpenGL-бэкендом, перезапуск обязателен. GPU не требуется.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y libosmesa6
%pip -q install mujoco==3.3.7 mediapy


In [ ]:
import os

# Обязательно до импорта mujoco или OpenGL.
os.environ["MUJOCO_GL"] = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"

import ctypes
import ctypes.util

library_path = ctypes.util.find_library("OSMesa")
print("Библиотека OSMesa:", library_path)

if library_path is None:
    raise RuntimeError("OSMesa не найдена. Повтори установку libosmesa6.")

ctypes.CDLL(library_path)

import numpy as np
import matplotlib.pyplot as plt
import mujoco
import mediapy as media
from time import perf_counter

print("MuJoCo:", mujoco.__version__)


# А. СЛАУ и равновесие упругой решётки
## А1. Реализация прямых методов
Реализуйте метод Гаусса с частичным выбором главного элемента, [LU с перестановками строк](https://ru.wikipedia.org/wiki/LUP-разложение) ($PA=LU$), метод Холецкого и треугольные решатели.

Функции не должны изменять исходные массивы. Предусмотрите диагностику вырожденных систем и неположительно определённых матриц.

`lu_factor` возвращает `(P, L, U)`; `cholesky_factor` — нижнетреугольную матрицу `L`.


In [ ]:
def forward_substitution(L, b):
    # TODO А1: решить нижнетреугольную систему.
    raise NotImplementedError("А1: прямой ход")


def backward_substitution(U, b):
    # TODO А1: решить верхнетреугольную систему.
    raise NotImplementedError("А1: обратный ход")


def gauss(A, b):
    # TODO А1: Гаусс с частичным выбором главного элемента.
    raise NotImplementedError("А1: метод Гаусса")


def lu_factor(A):
    # TODO А1: PA = LU. Учитывать перестановки в уже заполненной части L.
    raise NotImplementedError("А1: LU-разложение")


def lu_solve(factors, b):
    P, L, U = factors
    return backward_substitution(U, forward_substitution(L, P @ b))


def cholesky_factor(A):
    # TODO А1: проверить симметрию и реализовать A = L @ L.T.
    raise NotImplementedError("А1: разложение Холецкого")


def cholesky_solve(A, b):
    L = cholesky_factor(A)
    return backward_substitution(L.T, forward_substitution(L, b))


### Контрольные проверки А1
Ячейка проверяет решение, восстановление разложений и сохранение исходных данных. Добавьте собственные проверки вырожденной системы и отказа Холецкого для неподходящей матрицы. Прохождение этих проверок не заменяет эксперименты.


In [ ]:
rng = np.random.default_rng(42)
for size in (10, 50):
    B = rng.normal(size=(size, size))
    A = B + (size + 1)*np.eye(size)
    x_exact = rng.normal(size=size)
    b = A @ x_exact
    A_before, b_before = A.copy(), b.copy()
    P, L, U = lu_factor(A)
    np.testing.assert_allclose(P @ A, L @ U, atol=1e-9)
    for x in (gauss(A, b), lu_solve((P, L, U), b)):
        np.testing.assert_allclose(x, x_exact, rtol=1e-8, atol=1e-9)
    np.testing.assert_array_equal(A, A_before)
    np.testing.assert_array_equal(b, b_before)
    S = B.T @ B + np.eye(size)
    C = cholesky_factor(S)
    np.testing.assert_allclose(C @ C.T, S, atol=1e-9)
    np.testing.assert_allclose(cholesky_solve(S, S @ x_exact), x_exact,
                               rtol=1e-8, atol=1e-9)
A_pivot = np.array([[0., 2.], [1., 3.]])
b_pivot = A_pivot @ np.array([1., -2.])
np.testing.assert_allclose(gauss(A_pivot, b_pivot), [1., -2.])
np.testing.assert_allclose(lu_solve(lu_factor(A_pivot), b_pivot), [1., -2.])
print("Базовые проверки А1 пройдены.")


## 1. Упругая решётка: математическая модель

Рассмотрим решётку площадок, каждая из которых может перемещаться только вертикально. Соседние площадки соединены упругими связями. Каждая площадка также имеет пружинную опору относительно неподвижного основания.

Положительное перемещение $u_i$ направлено вниз.

Обозначения:
- $k_0$ — жёсткость опоры;
- $k_c$ — жёсткость связи соседей;
- $f_i$ — внешняя сила, направленная вниз;
- $N(i)$ — множество соседей площадки $i$.

Условие равновесия площадки:

$$
k_0u_i+\sum_{j\in N(i)}k_c(u_i-u_j)=f_i.
$$

Система уравнений всех площадок записывается в виде

$$
Ku=f.
$$

Элементы матрицы:

$$
K_{ii}=k_0+|N(i)|k_c,
$$

$$
K_{ij}=-k_c
$$

для соседних площадок; остальные внедиагональные элементы равны нулю.

При $k_0>0$ и $k_c>0$ матрица симметрична и положительно определена:

$$
u^\top Ku
=
k_0\sum_i u_i^2+
k_c\sum_{(i,j)\in E}(u_i-u_j)^2>0
$$

для любого ненулевого $u$. Каждая связь в множестве $E$ учитывается один раз.

При $k_0=0$ одинаковое перемещение всех площадок не деформирует связи. Матрица становится вырожденной.

In [ ]:
SIDE = 5                  # SIDE × SIDE площадок
SPACING = 0.22            # расстояние между центрами, м
HEIGHT = 0.65             # исходная высота, м
MASS = 0.15               # масса каждой площадки, кг

K_SUPPORT = 15.0          # жёсткость опоры каждого узла, Н/м
K_COUPLING = 40.0         # жёсткость связи соседей, Н/м
DAMPING = 0.6             # демпфирование каждого сустава, Н·с/м

n = SIDE**2

def index(row, col):
    return row * SIDE + col

xy = np.array([
    [(col - (SIDE - 1)/2) * SPACING,
     (row - (SIDE - 1)/2) * SPACING]
    for row in range(SIDE)
    for col in range(SIDE)
])

# Каждую связь включаем один раз.
edges = []
for row in range(SIDE):
    for col in range(SIDE):
        i = index(row, col)
        if col + 1 < SIDE:
            edges.append((i, index(row, col + 1)))
        if row + 1 < SIDE:
            edges.append((i, index(row + 1, col)))

# Опоры дают диагональную часть.
K = K_SUPPORT * np.eye(n)

# Связь i–j даёт энергию kc/2 * (ui - uj)^2.
for i, j in edges:
    K[i, i] += K_COUPLING
    K[j, j] += K_COUPLING
    K[i, j] -= K_COUPLING
    K[j, i] -= K_COUPLING

# Постоянная сила вниз на центральной площадке.
f = np.zeros(n)
loaded_node = index(SIDE // 2, SIDE // 2)
f[loaded_node] = 8.0

print("Количество неизвестных:", n)
print("Число обусловленности:", np.linalg.cond(K))
print("Минимальное собственное значение:", np.linalg.eigvalsh(K)[0])

plt.figure(figsize=(5, 4))
plt.imshow(K, cmap="coolwarm")
plt.colorbar(label="Жёсткость, Н/м")
plt.title("Матрица K")
plt.show()


## А2. Итерационные методы
Реализуйте Якоби и Гаусса–Зейделя. В Якоби все компоненты следующего приближения вычисляются по прежнему вектору; в Зейделе используются уже обновлённые компоненты.

Критерий останова:
$$\|Ax^{(k)}-b\|_2\leq \text{atol}+\text{rtol}\|b\|_2.$$

Функция возвращает решение и массив норм невязок, включая начальную. Число шагов равно `len(residuals)-1`. При исчерпании лимита или нечисловых значениях сообщайте об ошибке. `x0=None` означает нулевой старт.


In [ ]:
def iterative_solve(A, b, method="jacobi", x0=None,
                    rtol=1e-9, atol=1e-12, max_iter=20000):
    # TODO А2: Якоби и Зейдель, остановка и история невязки.
    # Не изменять входные массивы и x0.
    raise NotImplementedError("А2: итерационный решатель")


### Сравнение на одной нагрузке
Ниже используется библиотечное эталонное решение. Время одного короткого запуска не позволяет сделать надёжный вывод о скорости: для экспериментов используйте повторы и медиану времени.


In [ ]:
u_reference = np.linalg.solve(K, f)

results = {}

for name, solver in [
    ("Гаусс", gauss),
    ("LU", lambda A, b: lu_solve(lu_factor(A), b)),
    ("Холецкий", cholesky_solve),
]:
    start = perf_counter()
    u = solver(K, f)
    elapsed = perf_counter() - start
    results[name] = u

    print(
        f"{name:10s}: "
        f"невязка = {np.linalg.norm(K @ u - f):.2e}, "
        f"время = {elapsed*1000:.2f} мс"
    )

u_jacobi, r_jacobi = iterative_solve(K, f, "jacobi")
u_seidel, r_seidel = iterative_solve(K, f, "seidel")

results["Якоби"] = u_jacobi
results["Зейдель"] = u_seidel

for name, u in results.items():
    relative_error = (
        np.linalg.norm(u - u_reference)
        / np.linalg.norm(u_reference)
    )
    print(f"{name:10s}: относительное расхождение = {relative_error:.2e}")

plt.figure(figsize=(7, 4))
plt.semilogy(r_jacobi, label="Якоби")
plt.semilogy(r_seidel, label="Зейдель")
plt.xlabel("Номер итерации")
plt.ylabel("Норма невязки, Н")
plt.grid()
plt.legend()
plt.show()

print("Итераций Якоби:", len(r_jacobi) - 1)
print("Итераций Зейделя:", len(r_seidel) - 1)
print(f"Максимальный прогиб: {u_reference.max()*1000:.1f} мм")

# Это решение будет показано в MuJoCo красными метками.
u_static = results["Гаусс"].copy()


## А3. Проверка физической симуляцией
В MuJoCo каждая площадка имеет вертикальный сустав; `fixed tendon` задаёт пружину с координатой $u_i-u_j$. Гравитация и контакты отключены, силы задаются явно.

Динамика происходящего описывается уравнением: $M\ddot u+C\dot u+Ku=f$, где матрица $M$ скрывает инерционные свойства системы, а $C$ - скоростые взаимодействия. Так как мы рассматриваем систему в установившемся состоянии, получаем $Ku=f$.

Красные метки показывают вычисленное статическое положение; цветные площадки — физическое движение. Линии между площадками лишь визуализируют связи.

In [ ]:
body_parts = []

for i, (x, y) in enumerate(xy):
    color = "1 0.65 0.15 1" if f[i] != 0 else "0.15 0.55 0.85 1"

    body_parts.append(f"""
    <body name="node_{i}" pos="{x} {y} {HEIGHT}">
      <joint name="joint_{i}" type="slide"
             axis="0 0 -1"
             stiffness="{K_SUPPORT}"
             springref="0"
             damping="{DAMPING}"/>
      <geom name="plate_{i}" type="box"
            size="{SPACING*0.42} {SPACING*0.42} 0.025"
            mass="{MASS}" rgba="{color}"/>
    </body>
    """)

tendon_parts = []

for i, j in edges:
    tendon_parts.append(f"""
    <fixed name="spring_{i}_{j}"
           stiffness="{K_COUPLING}" springlength="0">
      <joint joint="joint_{i}" coef="1"/>
      <joint joint="joint_{j}" coef="-1"/>
    </fixed>
    """)

grid_xml = f"""
<mujoco model="elastic_grid">
  <option timestep="0.002"
          gravity="0 0 0"
          integrator="implicitfast"/>

  <visual>
    <global offwidth="640" offheight="480"/>
    <headlight ambient="0.45 0.45 0.45"/>
  </visual>

  <default>
    <geom contype="0" conaffinity="0"/>
  </default>

  <worldbody>
    <light pos="0 -1 3" dir="0 0 -1"/>
    <geom type="plane" size="3 3 0.1"
          rgba="0.92 0.94 0.97 1"/>
    {''.join(body_parts)}
  </worldbody>

  <tendon>
    {''.join(tendon_parts)}
  </tendon>
</mujoco>
"""

grid_model = mujoco.MjModel.from_xml_string(grid_xml)
grid_data = mujoco.MjData(grid_model)

body_ids = [
    mujoco.mj_name2id(grid_model, mujoco.mjtObj.mjOBJ_BODY, f"node_{i}")
    for i in range(n)
]

# Проверяем соответствие сил MuJoCo матрице K.
probe = mujoco.MjData(grid_model)
probe.qpos[:] = np.random.default_rng(42).normal(0, 0.01, n)
mujoco.mj_forward(grid_model, probe)

np.testing.assert_allclose(
    probe.qfrc_passive,
    -K @ probe.qpos,
    atol=1e-10,
)

print("Пружинные силы MuJoCo соответствуют матрице K.")


In [ ]:
GRID_FPS = 30
GRID_DURATION = 6.0
frame_times = np.arange(int(GRID_FPS * GRID_DURATION)) / GRID_FPS

camera = mujoco.MjvCamera()
camera.type = mujoco.mjtCamera.mjCAMERA_FREE
camera.lookat[:] = [0, 0, HEIGHT - 0.1]
camera.distance = max(2.3, SIDE * SPACING * 2.2)
camera.azimuth = 135
camera.elevation = -30


def add_sphere(scene, position, radius, rgba):
    if scene.ngeom >= scene.maxgeom:
        return

    mujoco.mjv_initGeom(
        scene.geoms[scene.ngeom],
        mujoco.mjtGeom.mjGEOM_SPHERE,
        np.array([radius, radius, radius]),
        np.asarray(position, dtype=float),
        np.eye(3).ravel(),
        np.asarray(rgba, dtype=np.float32),
    )
    scene.ngeom += 1


def add_segment(scene, start, end):
    if scene.ngeom >= scene.maxgeom:
        return

    geom = scene.geoms[scene.ngeom]
    mujoco.mjv_initGeom(
        geom,
        mujoco.mjtGeom.mjGEOM_CAPSULE,
        np.zeros(3),
        np.zeros(3),
        np.eye(3).ravel(),
        np.array([0.3, 0.3, 0.35, 1], dtype=np.float32),
    )
    mujoco.mjv_connector(
        geom,
        mujoco.mjtGeom.mjGEOM_CAPSULE,
        0.007,
        np.asarray(start, dtype=float),
        np.asarray(end, dtype=float),
    )
    scene.ngeom += 1


mujoco.mj_resetData(grid_model, grid_data)
mujoco.mj_forward(grid_model, grid_data)

grid_frames = []
time_log = []
error_log = []
residual_log = []
velocity_log = []

grid_renderer = mujoco.Renderer(
    grid_model, height=480, width=640,
    max_geom=max(1000, 5*n)
)

try:
    for frame_time in frame_times:
        while grid_data.time < frame_time - 1e-10:
            # Сила положительна по направлению slide-сустава:
            # в нашей модели это вниз.
            grid_data.qfrc_applied[:] = f
            mujoco.mj_step(grid_model, grid_data)

        grid_data.qfrc_applied[:] = f
        mujoco.mj_forward(grid_model, grid_data)

        time_log.append(grid_data.time)
        error_log.append(np.linalg.norm(grid_data.qpos - u_static))
        residual_log.append(np.linalg.norm(K @ grid_data.qpos - f))
        velocity_log.append(np.linalg.norm(grid_data.qvel))

        grid_renderer.update_scene(grid_data, camera=camera)
        scene = grid_renderer.scene

        # Связи соседей на верхней поверхности площадок.
        positions = np.array([
            grid_data.xpos[body_id].copy() for body_id in body_ids
        ])
        positions[:, 2] += 0.028

        for i, j in edges:
            add_segment(scene, positions[i], positions[j])

        # Красные метки рассчитанного статического положения.
        for i, (x, y) in enumerate(xy):
            add_sphere(
                scene,
                [x, y, HEIGHT - u_static[i] + 0.04],
                0.012,
                [1.0, 0.15, 0.2, 1.0],
            )

        grid_frames.append(grid_renderer.render().copy())
finally:
    grid_renderer.close()

media.show_video(grid_frames, fps=GRID_FPS)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, values, label in zip(axes,
        [error_log, residual_log, velocity_log],
        ["Расхождение с СЛАУ, м", "Невязка, Н", "Норма скорости, м/с"]):
    ax.semilogy(time_log, np.maximum(values, 1e-16))
    ax.set(xlabel="Время, с", ylabel=label)
    ax.grid()
plt.tight_layout()
plt.show()
print("Относительное расхождение:",
      np.linalg.norm(grid_data.qpos-u_static)/np.linalg.norm(u_static))


## А4. Численные эксперименты с СЛАУ
Для каждого эксперимента укажите параметры, представьте график или таблицу и объясните результат.

1. **Масштабируемость.** Решётки $5\times5$, $10\times10$, $20\times20$. Сравните время прямых методов и число итераций. Для больших задач видео строить необязательно.
2. **Несколько нагрузок.** Подготовьте не менее 10 правых частей при одной $K$. Отдельно измерьте время разложения и последующих решений для LU и Холецкого; сравните с повторным применением метода Гаусса.
3. **Начальный вектор.** Как начальные условия используйте нулевой, случайный единичной нормы и близкий к эталонному решению. Постройте ошибку и невязку по итерациям. Для графика ошибки добавьте сохранение приближений, не меняя основной интерфейс функции.
4. **Чувствительность.** Изменяйте $k_0$ при фиксированных размере и $k_c$. Сопоставьте число обусловленности $\nu(K)$, число итераций и изменение решения при малом возмущении $f$. Повторите для нескольких направлений возмущения.
5. **Свободная решётка.** Рассмотрите $k_0=0$. Объясните вырожденность и существование равновесия в зависимости от суммарной нагрузки. Не пытайтесь интерпретировать огромные числа от решателя как корректное единственное решение.

При изменении параметров заново собирайте матрицу и XML-модель. Для удобства можете вынести сборку решётки в функцию.


In [ ]:
# TODO А4: код экспериментов, графики и таблицы.


**Выводы А3–А4:** заполните здесь. Объясните увиденное, не только перечислите численные результаты.


## А5. Контрольные матрицы: обусловленность и расходимость
1. Для матриц Гильберта $H_{ij}=1/(i+j-1)$ размеров $5,7,9,12$ задайте $x_*=\mathbf1$, $b=Hx_*$. Сравните методы Гаусса, LU и Холецкого **на одной матрице $H$**. Вычислите ошибку, невязку и число обусловленности. Корректно зафиксируйте отказ разложения, если округления мешают его выполнению.
2. Отдельно рассмотрите $A=H^\top H$, $b=Ax^*$. Объясните, почему переход к этой матрице усиливает проблему обусловленности.
3. Для положительно определенной матрицы с преобладающей диагональю  $A=B^\top B+\alpha I$ (если $\alpha$ достаточно большое) при $n=100,300$ сравните прямые методы; при $n=500$ исследуйте методы Якоби и Зейделя.


In [ ]:
# TODO А5: контрольные численные эксперименты.


**Выводы А5:** …


# Б. Обратная кинематика двухзвенного механизма
Решатели методом Гаусса или LU из части А теперь используется внутри нелинейного метода.


## 1. Модель механизма

Механизм состоит из двух звеньев длиной $l_1$ и $l_2$. Точка подвеса находится на высоте $z_0$. Движение происходит в вертикальной плоскости $xz$.

Обобщённые координаты:
- $q_1$ — угол первого звена относительно вертикали вниз;
- $q_2$ — угол второго звена относительно первого.

При $q_1=q_2=0$ оба звена направлены вертикально вниз.

В модели присутствуют гравитация, инерция и сопротивление в суставах. Два привода позволяют задавать моменты сил.


In [ ]:
L1 = L2 = 0.7
BASE_Z = 1.7

arm_xml = f"""
<mujoco model="double_pendulum">
  <option timestep="0.002"
          gravity="0 0 -9.81"
          integrator="implicitfast"/>

  <visual>
    <global offwidth="640" offheight="480"/>
    <headlight ambient="0.5 0.5 0.5"/>
  </visual>

  <default>
    <joint damping="0.15" armature="0.01"/>
    <geom contype="0" conaffinity="0"/>
  </default>

  <worldbody>
    <light pos="0 -3 4" dir="0 1 -1"/>
    <camera name="front" pos="0 -4 1.1"
            xyaxes="1 0 0 0 0 1"/>

    <geom type="plane" size="3 3 0.1"
          rgba="0.92 0.94 0.97 1"/>

    <geom type="sphere" pos="0 0 {BASE_Z}"
          size="0.065" rgba="0.2 0.2 0.25 1"/>

    <body name="link1" pos="0 0 {BASE_Z}">
      <joint name="q1" type="hinge" axis="0 -1 0"/>
      <geom type="capsule"
            fromto="0 0 0 0 0 {-L1}"
            size="0.035" mass="1"
            rgba="0.15 0.45 0.9 1"/>

      <body name="link2" pos="0 0 {-L1}">
        <joint name="q2" type="hinge" axis="0 -1 0"/>
        <geom type="sphere" size="0.05"
              rgba="0.2 0.2 0.25 1"/>
        <geom type="capsule"
              fromto="0 0 0 0 0 {-L2}"
              size="0.03" mass="0.7"
              rgba="0.1 0.7 0.6 1"/>
        <site name="tip" pos="0 0 {-L2}"
              size="0.045" rgba="1 0.25 0.3 1"/>
      </body>
    </body>
  </worldbody>

  <actuator>
    <motor joint="q1" gear="1"/>
    <motor joint="q2" gear="1"/>
  </actuator>
</mujoco>
"""

arm_model = mujoco.MjModel.from_xml_string(arm_xml)
arm_data = mujoco.MjData(arm_model)
tip_id = mujoco.mj_name2id(
    arm_model, mujoco.mjtObj.mjOBJ_SITE, "tip"
)

print("Число степеней свободы:", arm_model.nv)


## 2. Прямая и обратная кинематика

Прямая кинематика определяет положение конца механизма по углам:

$$
p(q)=
\begin{pmatrix}
l_1\sin q_1+l_2\sin(q_1+q_2)\\
z_0-l_1\cos q_1-l_2\cos(q_1+q_2)
\end{pmatrix}.
$$

В обратной задаче известна целевая точка $p_*$, а углы требуется найти:

$$
F(q)=p(q)-p_*=0.
$$

Решение может быть не единственным. Кроме того, целевая точка может оказаться недостижимой.

Без ограничений на углы условие достижимости:

$$
|l_1-l_2|\leq \|p_*-(0,z_0)^\top\|_2\leq l_1+l_2.
$$

### Метод Ньютона

В окрестности текущего приближения $q_k$ используем линеаризацию:

$$
F(q_k+\Delta q_k)\approx F(q_k)+J(q_k)\Delta q_k,
$$

где $J$ — матрица Якоби. Приравнивая линейное приближение к нулю, получаем СЛАУ:

$$
J(q_k)\Delta q_k=-F(q_k).
$$

После её решения обновляем углы:

$$
q_{k+1}=q_k+\Delta q_k.
$$

Таким образом, каждая итерация Ньютона включает решение СЛАУ. Явно вычислять обратную матрицу не требуется.

Для двухзвенного механизма:

$$
J(q)=
\begin{pmatrix}
l_1\cos q_1+l_2\cos(q_1+q_2) & l_2\cos(q_1+q_2)\\
l_1\sin q_1+l_2\sin(q_1+q_2) & l_2\sin(q_1+q_2)
\end{pmatrix}.
$$


## Б1. Реализация кинематики и метода Ньютона
Реализуйте `endpoint`, аналитический `jacobian` и `inverse_kinematics`.

Реализуйте метод Ньютона, проверяйте невязку, ограничивайте число шагов, корректно обрабатывайте сингулярную систему и невозможность подобрать шаг.

Возвращайте `(q, iterations)`, где `iterations` — число сделанных обновлений. Для исследования сохраняйте историю невязки, поправки и шага в собственной диагностической функции. Не вычисляйте обратную матрицу явно.


In [ ]:
def endpoint(q):
    # TODO Б1: положение конца [x, z].
    raise NotImplementedError("Б1: прямая кинематика")


def jacobian(q):
    # TODO Б1: матрица частных производных 2×2.
    raise NotImplementedError("Б1: якобиан")


def solve_linear(A, b):
    return gauss(A, b)


def inverse_kinematics(target, q0, tol=1e-9, max_iter=40):
    # TODO Б1: метод Ньютона.
    # Возвращать (q, iterations), не изменять q0.
    raise NotImplementedError("Б1: метод Ньютона")


In [ ]:
q_test = np.array([0.4, 1.2])
arm_data.qpos[:] = q_test
mujoco.mj_forward(arm_model, arm_data)
np.testing.assert_allclose(endpoint(q_test),
                          arm_data.site_xpos[tip_id][[0, 2]], atol=1e-10)
h = 1e-6
J_fd = np.column_stack([
    (endpoint(q_test+h*np.eye(2)[j])-endpoint(q_test-h*np.eye(2)[j]))/(2*h)
    for j in range(2)
])
np.testing.assert_allclose(jacobian(q_test), J_fd, atol=1e-7)
q_found, count = inverse_kinematics(endpoint(q_test), [0.3, 1.1])
assert np.linalg.norm(endpoint(q_found)-endpoint(q_test)) < 1e-8
print("Базовые проверки Б1 пройдены.")


## 3. Расчёт заданных углов

Придумаем траекторию, чтобы манипулятор её исполнил. Сердечко задаётся параметрической кривой $p_*(t)$. Выбираем последовательность точек этой кривой и для каждой решаем обратную задачу кинематики.

Для первой точки задаём начальные углы вручную. Для следующих используем решение предыдущей задачи:

$$
q^{(0)}_{i+1}=q_i.
$$

При близких соседних точках это обычно сокращает число итераций и помогает сохранять одну ветвь решения.

На этом этапе мы рассчитываем **заданные углы**, а не физическое движение механизма.

Проверяем:
- достигнутую невязку;
- число итераций;
- непрерывность полученной последовательности углов.


In [ ]:
def heart(phase):
    """Сердечко целиком внутри рабочей области."""
    x = 16*np.sin(phase)**3
    z = (
        13*np.cos(phase)
        - 5*np.cos(2*phase)
        - 2*np.cos(3*phase)
        - np.cos(4*phase)
    )
    return np.array([
        0.45 + 0.018*x,
        0.95 + 0.018*z,
    ])

ARM_FPS = 30
ARM_DURATION = 12.0
N = int(ARM_FPS * ARM_DURATION)

times = np.arange(N + 1) / ARM_FPS
phases = 2*np.pi * times / ARM_DURATION
targets = np.array([heart(phase) for phase in phases])

q_reference = []
iteration_counts = []

q_guess = np.array([-0.4, 1.7])

for target in targets:
    q_guess, iterations = inverse_kinematics(target, q_guess)
    q_reference.append(q_guess.copy())
    iteration_counts.append(iterations)

q_reference = np.array(q_reference)
dq_reference = np.gradient(q_reference, 1/ARM_FPS, axis=0)

ik_errors = np.linalg.norm(
    np.array([endpoint(q) for q in q_reference]) - targets,
    axis=1,
)

print(f"Максимальная ошибка IK: {ik_errors.max():.2e} м")
print(f"Максимум итераций Ньютона: {max(iteration_counts)}")

plt.figure(figsize=(6, 5))
plt.plot(targets[:, 0], targets[:, 1], color="hotpink")
plt.scatter(0, BASE_Z, color="black", label="Подвес")
plt.axis("equal")
plt.xlabel("x, м")
plt.ylabel("z, м")
plt.legend()
plt.grid()
plt.show()


## 4. Физическое воспроизведение траектории

Рассчитанные углы становятся заданием для PD-регулятора:

$$
\tau=
K_P(q_{\mathrm{зад}}-q)
+
K_D(\dot q_{\mathrm{зад}}-\dot q)
+
b(q,\dot q),
$$

где $b(q,\dot q)$ компенсирует гравитационные и скоростные силы модели.

Регулятор задаёт моменты в суставах, а MuJoCo рассчитывает движение.

На видео:
- розовые точки — заданная траектория;
- красная точка — текущая цель;
- зелёный след — фактическое движение конца механизма.

Промежуточные итерации Ньютона на этом видео не показаны: обратная кинематика рассчитана заранее.


In [ ]:
def reference_at(t):
    """Линейная интерполяция заданных углов и скоростей."""
    t = np.clip(t, 0.0, ARM_DURATION)

    q = np.array([
        np.interp(t, times, q_reference[:, j])
        for j in range(2)
    ])

    dq = np.array([
        np.interp(t, times, dq_reference[:, j])
        for j in range(2)
    ])

    return q, dq


def add_dot(scene, point, radius, color):
    if scene.ngeom >= scene.maxgeom:
        return

    geom = scene.geoms[scene.ngeom]
    mujoco.mjv_initGeom(
        geom,
        mujoco.mjtGeom.mjGEOM_SPHERE,
        np.array([radius, radius, radius]),
        np.asarray(point, dtype=float),
        np.eye(3).ravel(),
        np.asarray(color, dtype=np.float32),
    )
    scene.ngeom += 1


mujoco.mj_resetData(arm_model, arm_data)
arm_data.qpos[:] = q_reference[0]
arm_data.qvel[:] = dq_reference[0]
mujoco.mj_forward(arm_model, arm_data)

KP = np.array([180.0, 100.0])
KD = np.array([25.0, 12.0])

arm_frames = []
actual_points = []

arm_renderer = mujoco.Renderer(
    arm_model, height=480, width=640, max_geom=3000
)

try:
    for frame_time in times[:-1]:
        while arm_data.time < frame_time - 1e-10:
            q_desired, dq_desired = reference_at(arm_data.time)

            # qfrc_bias компенсирует гравитацию и скоростные силы.
            arm_data.ctrl[:] = (
                KP * (q_desired - arm_data.qpos)
                + KD * (dq_desired - arm_data.qvel)
                + arm_data.qfrc_bias
            )
            mujoco.mj_step(arm_model, arm_data)

        # Обновляем положения site для текущего состояния.
        mujoco.mj_forward(arm_model, arm_data)
        actual_points.append(arm_data.site_xpos[tip_id].copy())

        arm_renderer.update_scene(arm_data, camera="front")
        scene = arm_renderer.scene

        # Заданная траектория: розовые точки.
        for x, z in targets[::3]:
            add_dot(
                scene, [x, -0.025, z], 0.005,
                [1.0, 0.45, 0.65, 1.0]
            )

        # След реального конца маятника.
        for point in actual_points:
            p = point.copy()
            p[1] = -0.045
            add_dot(
                scene, p, 0.007,
                [0.1, 0.75, 0.3, 1.0]
            )

        # Текущая цель.
        target = heart(2*np.pi * frame_time / ARM_DURATION)
        add_dot(
            scene, [target[0], -0.07, target[1]],
            0.025, [1.0, 0.15, 0.2, 1.0]
        )

        arm_frames.append(arm_renderer.render().copy())
finally:
    arm_renderer.close()

actual_points = np.array(actual_points)

media.show_video(arm_frames, fps=ARM_FPS)


## 5. Анализ ошибок

Различаем две ошибки.

**Ошибка обратной кинематики:**

$$
e_{\mathrm{IK}}(t)=
\|p(q_{\mathrm{зад}}(t))-p_*(t)\|_2.
$$

Она характеризует точность найденных углов относительно целевой точки.

**Ошибка физического слежения:**

$$
e_{\mathrm{track}}(t)=
\|p(q_{\mathrm{факт}}(t))-p_*(t)\|_2.
$$

Она зависит от динамики механизма, регулятора и скорости движения.

Даже при точном решении обратной кинематики фактическая траектория может отличаться от заданной.

### Сингулярные конфигурации

Для нашего якобиана

$$
\det J(q)=l_1l_2\sin q_2.
$$

Когда звенья лежат на одной прямой, якобиан вырождается. Вблизи таких конфигураций небольшое изменение цели может требовать большого изменения углов.

Исследуйте, как приближение к сингулярности влияет на поправку Ньютона, число итераций и чувствительность решения.


In [ ]:
actual_xz = actual_points[:, [0, 2]]
tracking_error = np.linalg.norm(
    actual_xz - targets[:-1], axis=1
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(
    targets[:, 0], targets[:, 1],
    "--", color="hotpink", label="Задание"
)
axes[0].plot(
    actual_xz[:, 0], actual_xz[:, 1],
    color="seagreen", label="MuJoCo"
)
axes[0].set_aspect("equal")
axes[0].set_xlabel("x, м")
axes[0].set_ylabel("z, м")
axes[0].legend()
axes[0].grid()

axes[1].plot(times[:-1], tracking_error * 1000)
axes[1].set_xlabel("Время, с")
axes[1].set_ylabel("Ошибка слежения, мм")
axes[1].grid()

plt.tight_layout()
plt.show()

print(f"Средняя ошибка: {tracking_error.mean()*1000:.2f} мм")
print(f"Максимальная ошибка: {tracking_error.max()*1000:.2f} мм")


## Б2–Б3. Исследование обратной кинематики
**Б2. Сходимость.** Для одной достижимой цели постройте карту по начальным углам $q_1,q_2\in[-\pi,\pi]$ (не менее $30\times30$ точек). Покажите число итераций, ветвь решения и неудачные старты. Углы сравнивайте с учётом периодичности $2\pi$.

Исследуйте недостижимую цель и цели около сингулярности. Постройте истории невязки, нормы поправки и числа обусловленности якобиана. Отличайте недостижимость от неудачи алгоритма при конкретном старте.

**Б3. Траектория.** Создайте собственную достижимую траекторию и воспроизведите её. Сравните минимум два времени обхода. Разделите ошибку IK в узлах траектории, погрешность интерполяции между узлами и ошибку физического слежения. Красная цель на видео должна соответствовать вашей новой траектории: замените `heart` в обоих местах или используйте единую функцию траектории.

Для исследования скорости сходимости используйте цель с невырожденным якобианом и достаточно близкий старт; квадратичная сходимость является локальным свойством.


In [ ]:
# TODO Б2–Б3: эксперименты и собственная траектория.


**Выводы Б:** …


# В. Сдача
Сдайте заполненный `.ipynb` с реализациями, воспроизводимыми экспериментами, графиками и выводами. Видео можно не сохранять внутри файла: оставьте код их получения. Случайные генераторы должны иметь фиксированный seed.

Перед сдачей перезапустите сеанс и выполните ячейки последовательно. Уберите отладочный вывод и убедитесь, что обязательные TODO заполнены. Ожидаемые отказы в экспериментах перехватывайте и подписывайте; они не должны обрывать весь блокнот.
